# 01 — Raw data analysis (Step 1)
**Team 8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**

Goal of this notebook (paper step 1 — *raw data visualization/analysis*): load the **ACC Arena** raw
data, build a tidy per-user/per-timestamp table, and inspect it to decide which features look promising
for predicting **throughput**.

> The raw dataset is in **wide** format: one folder per metric (Throughput, BLER, PRB, RU, SINR DL/UL,
> Positions), each CSV holding `time` + one column per user (`entityStats id N`), ~500 users per file.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 60            # downsampling step in seconds. Lower = more samples, more time (try 30 or 15).
N_USERS          = 1500          # ACC Arena users to load. Closest-users neighbours are searched WITHIN this subset.
N_USERS_SALT     = 1000          # Salt&Tar users (transfer-learning notebook)
X_VALUES         = [1, 3, 5, 10] # number of closest users to experiment with
BEST_X           = 5             # X used for the transfer-learning experiment (pick the best from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw "wide" format -> tidy per-user/per-timestamp table) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def metric_files(venue_dir, subdir_glob, file_glob, n_files=None):
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)),
                   key=lambda p: int(re.search(r'_(\d+)_\d+\.csv$', p).group(1)))
    return files[:n_files] if n_files else files

def build_grid(ref_file, resample_seconds):
    t = pd.read_csv(ref_file, usecols=[0]).iloc[:, 0].values.astype(float)
    return np.arange(t[0], t[-1] + 1, resample_seconds)

def _align(df, grid, tol):
    df = df[~df.index.duplicated(keep="first")]
    return df.reindex(grid, method="nearest", tolerance=tol)

def load_metric(files, value_name, grid, tol, user_ids):
    out = []
    for f in files:
        df = pd.read_csv(f).rename(columns={pd.read_csv(f, nrows=0).columns[0]: "time"}).set_index("time")
        cmap = {c: int(re.search(r'(\d+)', c).group(1)) for c in df.columns}
        df = df[[c for c, u in cmap.items() if u in user_ids]].rename(columns=cmap)
        df = _align(df, grid, tol)
        out.append(df.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, grid, tol, user_ids):
    """Positions are interleaved blocks of (id, lat, lon, alt, mobileState).
    lat/lon are converted to local metres; mobileState is the traffic_type."""
    frames = []
    for f in files:
        df = pd.read_csv(f)
        t = df.iloc[:, 0].values.astype(float)
        arr = df.values.astype(float)
        ids = arr[0, 1::5].astype(int)
        lat, lon, alt, mob = arr[:, 2::5], arr[:, 3::5], arr[:, 4::5], arr[:, 5::5]
        lat0, lon0 = np.nanmean(lat), np.nanmean(lon)
        R = 6371000.0
        x = R * np.radians(lon - lon0) * math.cos(math.radians(lat0))
        y = R * np.radians(lat - lat0)
        def mk(v, name):
            d = pd.DataFrame(v, index=t, columns=ids)
            d = d[[u for u in ids if u in user_ids]]
            d.index.name = "time"
            return _align(d, grid, tol).reset_index().melt(id_vars="time", var_name="user_id", value_name=name)
        m = mk(x, "x").merge(mk(y, "y"), on=["time", "user_id"]) \
                      .merge(mk(alt, "z"), on=["time", "user_id"]) \
                      .merge(mk(mob, "traffic_type"), on=["time", "user_id"])
        frames.append(m)
    return pd.concat(frames, ignore_index=True)

def assemble(venue_key, n_users, resample_seconds):
    """Return a tidy DataFrame with one row per (user, timestamp)."""
    venue = find_venue_dir(DATA_ROOT, venue_key)
    user_ids = set(range(n_users))
    n_files = math.ceil(n_users / 500)
    tol = resample_seconds / 2
    ref = metric_files(venue, "*Throughput*", "*.csv", 1)[0]
    grid = build_grid(ref, resample_seconds)
    mf = lambda sub, fg: metric_files(venue, sub, fg, n_files)
    parts = [
        load_metric(mf("*Throughput*",     "*.csv"),    "throughput", grid, tol, user_ids),
        load_metric(mf("*BLER*",           "*.csv"),    "bler",       grid, tol, user_ids),
        load_metric(mf("*PRB*",            "*.csv"),    "prb",        grid, tol, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"),    "ru_id",      grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRDL_*.csv"), "sinr_dl", grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRUL_*.csv"), "sinr_ul", grid, tol, user_ids),
        load_positions(mf("*Positions*",   "*.csv"), grid, tol, user_ids),
    ]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p, on=["time", "user_id"], how="inner")
    df = df.dropna().reset_index(drop=True)
    df["user_id"]      = df["user_id"].astype(int)
    df["traffic_type"] = df["traffic_type"].round().astype(int)
    df["ru_id"]        = df["ru_id"].round().astype(int)
    return df

## Load a tidy sample of ACC Arena
We load a subset of users (fast) and downsample to one sample per minute.

In [ ]:
df = assemble("acc", n_users=min(N_USERS, 500), resample_seconds=RESAMPLE_SECONDS)
print("tidy shape:", df.shape)
df.head()

In [ ]:
df.describe()

## Throughput distribution and extreme tail
Throughput is heavily skewed. How rare — and how heavy — are the extreme samples?

In [ ]:
import matplotlib.pyplot as plt
act = df[df.traffic_type >= 2]["throughput"]           # active users only (idle is 0 by definition)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(act, bins=60); ax[0].set_title("Throughput, active users (Mbps)"); ax[0].set_xlabel("Mbps")
ax[1].hist(act[act <= np.percentile(act, 99)], bins=60, color="orange")
ax[1].set_title("Same, within the 99th percentile"); ax[1].set_xlabel("Mbps")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_throughput_dist.png", dpi=120); plt.show()

print("percentiles (Mbps):", {p: round(float(np.percentile(act, p)), 2) for p in [50, 90, 95, 99, 99.9, 100]})
dev = (act - act.mean())**2
for th in [np.percentile(act, 99), 100]:
    m = act > th
    print(f"samples > {th:6.1f} Mbps: {m.mean()*100:5.2f}% of data, carrying {dev[m].sum()/dev.sum()*100:4.1f}% of total variance")

The distribution is extremely heavy-tailed: the median is below 1 Mbps while the maximum reaches hundreds of
Mbps, and **the ~1% most extreme samples alone carry the majority of the total variance**. Squared-error
training and R² would be dominated by this handful of rare samples. → In preprocessing (notebook 02) we drop
samples above the 99th train-percentile (`OUTLIER_PCT`), restricting the regression to typical operating
conditions — and we state this explicitly when reporting results.

## Throughput by traffic type
Traffic type (0=off … 5=http) strongly conditions throughput.

In [ ]:
labels = {0:"off",1:"idle",2:"const",3:"video",4:"gaming",5:"http"}
groups = [df.loc[df.traffic_type==t, "throughput"] for t in sorted(df.traffic_type.unique())]
plt.figure(figsize=(8,4))
plt.boxplot(groups, labels=[labels.get(t,t) for t in sorted(df.traffic_type.unique())], showfliers=False)
plt.ylabel("Throughput (Mbps)"); plt.title("Throughput by traffic type")
plt.savefig(f"{RESULTS_DIR}/figures/01_throughput_by_traffic.png", dpi=120); plt.show()

## Feature correlations
Which numeric features move with throughput?

In [ ]:
num = ["throughput","bler","prb","sinr_dl","sinr_ul","x","y","z"]
corr = df[num].corr()
plt.figure(figsize=(7,6)); plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(num)), num, rotation=45, ha="right"); plt.yticks(range(len(num)), num)
for i in range(len(num)):
    for j in range(len(num)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(); plt.title("Correlation matrix"); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/01_corr.png", dpi=120); plt.show()

## SINR vs throughput, and user positions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,5))
s = df.sample(min(5000, len(df)), random_state=RANDOM_SEED)
sc = ax[0].scatter(s.sinr_dl, s.throughput, c=s.traffic_type, cmap="tab10", s=6, alpha=.5)
ax[0].set_xlabel("SINR DL (dB)"); ax[0].set_ylabel("Throughput (Mbps)"); ax[0].set_title("SINR vs throughput")
one_t = df[df.time == df.time.iloc[0]]
ax[1].scatter(one_t.x, one_t.y, c=one_t.traffic_type, cmap="tab10", s=10)
ax[1].set_xlabel("x (m)"); ax[1].set_ylabel("y (m)"); ax[1].set_title("User positions @ t0")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_sinr_positions.png", dpi=120); plt.show()

## What the data suggests
- **Regression target** `throughput` is non-negative and strongly right-skewed.
- **`idle` users have throughput ≈ 0 by definition** (and are ~half of all rows): a degenerate target the model
  would predict trivially from the traffic-type flag. → In notebook 02 we set `ACTIVE_ONLY=True` to regress
  only on **active** users (`traffic_type >= 2`). The extreme tail is handled with `OUTLIER_PCT` (notebook 02).
- **`traffic_type`** is the dominant driver → must be an input feature (one-hot).
- **`sinr_dl`/`sinr_ul`, `prb`, `bler`** are physical-layer indicators expected to correlate with throughput.
- Users are **spatially clustered** → the *features of the X closest users* (Team-8 feature) should add
  context about local cell load / interference. Built in notebook **02**.
- Both Neural Network and Random Forest are reasonable; non-linear relations + mixed feature types favour them.